# ENRICH-Aerial — GCP-controlled bundle adjustment

Full pipeline on the `nadir-2` sub-block: COLMAP structure from motion, then a
bundle adjustment that adds the surveyed ground control points (and optionally
the camera centres) to COLMAP's own Ceres problem.

The machinery lives in the `pycolmap_ba` package — `frontend` for the pycolmap
stages, `backend` for the extra residuals, `utils` for scoring and export — so
the cells below are the *application*: data, settings, and reading the numbers.
The maths is written up in [`docs/gcp_bundle_adjustment.md`](../docs/gcp_bundle_adjustment.md).


In [1]:
from pathlib import Path

from pycolmap import logging

from pycolmap_ba import backend, frontend, utils

import enrich_data

# --- paths ------------------------------------------------------------------
NADIR = enrich_data.DATA_ROOT / "nadir-2"
image_path = NADIR / "images"
database_path = NADIR / "colmap/database.db"
sfm_free_path = NADIR / "colmap/sparse"  # free network, as the mapper leaves it
sfm_gcp_path = NADIR / "colmap/sparse_gcp"  # after the controlled bundle

# --- what to run ------------------------------------------------------------
# Feature extraction and brute-force matching take a while on CPU, so by default
# an existing database is reused.
FORCE_OVERWRITE_DB = False
# Give the *mapper* the camera centres as pose priors. False leaves the free
# network in an arbitrary frame, which the Sim3 below then fixes.
USE_MAPPER_POSE_PRIORS = False
# Constrain the camera centres in the GCP *bundle*, like Metashape's _GT project
# (all 39 references enabled). The camera-centre accuracy row then becomes a fit
# to data the bundle was given - the honest configuration is False.
USE_CAMERA_PRIORS = True
# Let the controlled bundle free the focal length. Safe here, and only here: the
# control is what breaks the flat-block degeneracy.
REFINE_INTRINSICS = True

logging.set_log_destination(logging.INFO, Path.cwd() / "INFO.log")

## 1. Data

`load()` reads the CSVs as they are; `add_noise()` returns another object of the
same type with a simulated survey applied. `truth` is kept for scoring, `data`
is what the pipeline is allowed to see.

In [2]:
truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = truth  # or: enrich_data.add_noise(truth) to simulate a real survey
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, marker_name, u, v
camera_centers = data.camera_xyz
marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

# --- assumed accuracies (see docs: only the ratios matter) ------------------
tie_points_sigma_px = enrich_data.tie_points_sigma_px
gcp_sigma_px = enrich_data.gcp_sigma_px
gcp_sigma_m = enrich_data.gcp_sigma_m
camera_position_sigma_m = enrich_data.camera_position_sigma_m

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

156 marker image marks, 26 markers, 39 images
control: 18 points | check: 8 points (gcp.001, gcp.004, gcp.010, gcp.013, gcp.016, gcp.019, gcp.022, gcp.025)
survey noise: xy 2 cm, z 3 cm; marks 1.0 px
GCPs: ['gcp.002', 'gcp.003', 'gcp.005', 'gcp.006', 'gcp.007', 'gcp.008', 'gcp.009', 'gcp.011', 'gcp.012', 'gcp.014', 'gcp.015', 'gcp.017', 'gcp.018', 'gcp.020', 'gcp.021', 'gcp.023', 'gcp.024', 'gcp.026']
CP   ['gcp.001', 'gcp.004', 'gcp.010', 'gcp.013', 'gcp.016', 'gcp.019', 'gcp.022', 'gcp.025']


In [3]:
# Everything the reports need to score the block. Partial truth is fine: each
# field just enables one more row.
gt = utils.GroundTruth(
    camera_xyz=truth.camera_xyz,
    marker_xyz=truth.marker_xyz,
    control_names=truth.control_names,
    check_names=truth.check_names,
    cam_from_world=enrich_data.cam_from_world_all(truth),
    focal_px=enrich_data.FOCAL_PX,
)

## 2. Free network

Features, matches, known intrinsics, incremental mapping. Nothing here knows
about the GCPs yet.

In [4]:
frontend.extract_and_match(
    database_path,
    image_path,
    # SIMPLE_PINHOLE has a SINGLE focal parameter. With PINHOLE, fx and fy
    # refine independently and on this flat nadir block they diverged to
    # fx=3227.6 / fy=3818.1 - an 18% anisotropic stretch.
    # That is not a similarity, so no alignment can undo it and the whole block
    # ends up ~8.5 m out.
    # The renders have no lens distortion (README), so a distortion-free model
    # with a single focal is the correct choice - and is what Metashape used.
    frontend.FeatureOptions(camera_model="SIMPLE_PINHOLE", max_num_features=8192),
    overwrite=FORCE_OVERWRITE_DB,
)


Feature extraction and matching


W20260813 20:31:50.933662 700220 feature_extraction.cc:445] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20260813 20:31:50.933900 700832 feature_extraction.cc:494] === Feature extraction ===
I20260813 20:31:50.934368 700849 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 20:31:50.934388 700850 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 20:31:50.934432 700851 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 20:31:50.934456 700853 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 20:31:50.934465 700852 sift.cc:765] Creating SIFT CPU feature extractor
I20260813 20:31:50.934484 700854 sift.cc:765] Creating SIFT 

In [5]:
# The rendered focal is known exactly, so seed it instead of estimating it.
frontend.seed_intrinsics(
    database_path,
    frontend.CameraIntrinsics(
        width=enrich_data.IMAGE_WIDTH,
        height=enrich_data.IMAGE_HEIGHT,
        params=[enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT],
    ),
)


Seeding the known intrinsics
  camera 1: SIMPLE_PINHOLE [5865.1810585 3008.        2008.       ]


1

In [6]:
if USE_MAPPER_POSE_PRIORS:
    frontend.write_pose_priors(
        database_path, camera_centers, sigma_m=camera_position_sigma_m
    )

In [7]:
mappingOptions = frontend.MappingOptions(
    use_prior_position=USE_MAPPER_POSE_PRIORS,
    # A flat nadir block cannot determine focal length from imagery: focal
    # trades against scene depth. Hold the intrinsics at the known values;
    # the GCP bundle can free them later, where the control breaks the
    # degeneracy.
    refine_focal_length=False,
    # COLMAP's own tie-point filter defaults, restated so the tuning surface
    # is visible. All three are NO-OPS on this block - the 80/60 overlap
    # gives strong geometry: measured triangulation angle p1 = 7.4 deg,
    # median 26.8 deg, zero points below 5 deg, max reprojection 3.98 px -
    # but they matter on weaker blocks.
    filter_max_reproj_error=4.0,
    filter_min_tri_angle_deg=1.5,
    ba_local_min_tri_angle_deg=6.0,
)

rec = frontend.run_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=sfm_free_path,
    options=mappingOptions,
    seed=0,
    progress=True,
    write_text=True,
)


Incremental mapping

Images registered:   0%|                               |  0/39 [00:00<?, 0.00/s]

I20260813 20:31:51.011214 700220 incremental_pipeline.cc:348] Loading database
I20260813 20:31:51.011286 700220 database_cache.cc:73] Loading rigs...
I20260813 20:31:51.011324 700220 database_cache.cc:83]  1 in 0.000s
I20260813 20:31:51.011337 700220 database_cache.cc:91] Loading cameras...
I20260813 20:31:51.011353 700220 database_cache.cc:109]  1 in 0.000s
I20260813 20:31:51.011359 700220 database_cache.cc:117] Loading frames...
I20260813 20:31:51.011409 700220 database_cache.cc:127]  39 in 0.000s
I20260813 20:31:51.011415 700220 database_cache.cc:135] Loading matches...
I20260813 20:31:51.017874 700220 database_cache.cc:140]  441 in 0.006s
I20260813 20:31:51.017910 700220 database_cache.cc:148] Loading images...
I20260813 20:31:51.029668 700220 database_cache.cc:245]  39 in 0.012s (connected 39, loaded 39)
I20260813 20:31:51.029709 700220 database_cache.cc:259] Loading pose priors...
I20260813 20:31:51.029739 700220 database_cache.cc:267]  39 in 0.000s
I20260813 20:31:51.029747 7002

Images registered:   5%|█▍                         |  2/39 [00:00<00:09, 4.01/s]

I20260813 20:31:51.507728 700220 incremental_pipeline.cc:607] Registering image #24 (num_reg_frames=2)
I20260813 20:31:51.507772 700220 incremental_pipeline.cc:610] => Image sees 1783 / 7815 points
I20260813 20:31:51.768795 700220 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment
I20260813 20:31:51.939551 700220 incremental_pipeline.cc:607] Registering image #19 (num_reg_frames=3)
I20260813 20:31:51.939589 700220 incremental_pipeline.cc:610] => Image sees 3534 / 7315 points


Images registered:   8%|██▏                        |  3/39 [00:01<00:11, 3.23/s]

...

Images registered:  97%|██████████████████████████▎| 38/39 [00:29<00:01, 1.32/s]

I20260813 20:32:19.893235 700220 incremental_pipeline.cc:607] Registering image #39 (num_reg_frames=38)
I20260813 20:32:19.893268 700220 incremental_pipeline.cc:610] => Image sees 3682 / 5529 points


Images registered: 100%|███████████████████████████| 39/39 [00:30<00:00, 1.32/s]

I20260813 20:32:20.543340 700220 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment
I20260813 20:32:21.661906 700220 incremental_pipeline.cc:87] Retriangulation and Global bundle adjustment
I20260813 20:32:22.773262 700220 incremental_pipeline.cc:793] Keeping successful reconstruction
I20260813 20:32:22.823225 700220 timer.cc:90] Elapsed time: 0.529 [minutes]


Images registered: 100%|███████████████████████████| 39/39 [00:30<00:00, 1.32/s]Reconstruction:
	num_rigs = 1
	num_cameras = 1
	num_frames = 39
	num_reg_frames = 39
	num_images = 39
	num_points3D = 43749
	num_observations = 221660
	mean_track_length = 5.06663
	mean_observations_per_image = 5683.59
	mean_reprojection_error = 0.607376
  camera: [5865.1810585 3008.        2008.       ]



In [ ]:
utils.banner("Free network vs the known camera centres")
rmse = utils.camera_center_rmse(rec, camera_centers)
print(f"camera centre RMSE: {rmse:.4f} m")

# A bundle adjustment is a local optimiser: if the network sits in an arbitrary
# frame, the GCP residuals start hopeless and the solve diverges.
# A least-squares similarity over all 39 centres brings it close enough.
if rmse > 1.0:
    utils.align_to_positions(rec, camera_centers)
    print(f"after Sim3: {utils.camera_center_rmse(rec, camera_centers):.4f} m")

    # Overwrite the free network with the aligned version
    utils.export_reconstruction(rec, sfm_free_path)

# A similarity absorbs uniform scale but NOT anisotropic distortion, so a large
# residual that survives the alignment means the free network is deformed -
# almost always a runaway self-calibration - and no control will rescue it.


Free network vs the known camera centres
camera centre RMSE: 0.0155 m


In [15]:
# Scored before the control enters, so the effect of the GCPs is measurable.
# No adjusted marker coordinates exist yet, hence the empty dict: only the poses
# and the intrinsics can be scored at this point.
utils.block_report(rec)
utils.accuracy_report(rec, {}, gt, "free network")


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.135 3008.    2008.   ]
  points3D          43,749      observations 221,660
  track length      mean 5.07   median 4
                    2: 1,114 (2.5%)   3-5: 30,330 (69.3%)   6-9: 8,254 (18.9%)   10+: 4,051 (9.3%)
  observations/img  min 3,414   mean 5,684   max 7,026

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.816   p90 1.456   p99 2.261   max 3.98
    > 2 px: 1,109 points (2.5%)

  redundancy        443,320 measurements - 131,484 unknowns  ->  r = 311,836

ACCURACY vs GROUND TRUTH - free network
                          bias E   bias N   bias U    RMSE 3D
                            (cm)     (cm)     (cm)       (cm)
  camera centres n=39      +0.10    -0.01    +0.00       1.72   

  camera rotations       mean 0.0000 deg   max 0.0000 deg
  focal                  5865.135  vs reference 5865.181  (-0.046)


## 3. GCP-controlled bundle adjustment

Each marker becomes a parameter block of its own, tied to the poses by its image
marks and - for the control points only - to the survey by a Gaussian prior. The
check points get rays but no prior, so their residual is the one unbiased
accuracy figure in the whole notebook.

All sigmas are given in native units (px, m); the backend expresses them in
tie-point sigma units, because COLMAP adds its tie-point residuals unweighted.

In [12]:
solverOptions = backend.SolverOptions(
    refine_focal_length=REFINE_INTRINSICS,
    # Points with fewer rays would be IGNORED by the solver, not deleted:
    # they would keep pre-bundle coordinates and still reach the export.
    # 0 = disabled; the post-solve filter below is the one that deletes.
    min_track_length=0,
    # COLMAP's global-BA default. SOFT_L1 @ 1.0 would instead down-weight the
    # reprojection tail (p95 1.77, p99 2.27, max 3.98 px) without discarding
    # redundancy - the soft alternative to hard filtering.
    loss_function_type=None,
    # COLMAP ships function_tolerance = 0, which *disables* the relative
    # stopping test and makes every solve run to the iteration cap and report
    # NO_CONVERGENCE although the cost stopped improving. 1e-8 fixes it.
    function_tolerance=1e-8,
)
markersConstraints = backend.MarkerConstraints(
    observations=marker_data,
    xyz=surveyed_xyz,
    control_names=control_names,  # the check points stay unconstrained
    sigma_px=gcp_sigma_px,
    sigma_m=gcp_sigma_m,
    robust_loss_scale=2.8,  # Cauchy on the marks, to absorb mis-identification
)
camera_posesConstraints = (
    backend.CameraPoseConstraints(
        positions=camera_centers, sigma_m=camera_position_sigma_m
    )
    if USE_CAMERA_PRIORS
    else None
)
filterOptions = backend.FilterOptions(
    max_reproj_error=4.0, min_tri_angle_deg=1.5, min_track_length=0
)

result = backend.run_bundle_adjustment(
    rec,
    solver=solverOptions,
    markers=markersConstraints,
    camera_poses=camera_posesConstraints,
    # Post-solve filtering, same thresholds as the mapper: no-ops here, kept
    # explicit so every threshold in the notebook is visible in one place.
    filters=filterOptions,
    tie_points_sigma_px=tie_points_sigma_px,
    covariance=True,
    verbose=True,
)
marker_xyz = result.marker_xyz  # adjusted marker coordinates

tie points      : sigma 0.5 px  (reference unit)
marker image obs: sigma 1.0 px -> 2.00 in tie units
marker world    : sigma [0.02 0.02 0.03] m -> [0.04 0.04 0.06] in tie units
camera positions: sigma 0.1 m -> 0.2 in tie units
gauge           : None
  problem: 221,873 blocks = 221,660 tie points + 213 added (156 marker image, 18 marker world, 39 camera pose)
  solve 1: BundleAdjustmentTerminationType.CONVERGENCE
  filtered 0 observations -> 43,749 points remain


In [13]:
cov = result.covariance
cov.get_point_cov(1)

array([[ 5.69206836e-04, -3.09487799e-05,  8.35570458e-04],
       [-3.09487799e-05,  2.88590589e-04, -9.10527452e-05],
       [ 8.35570458e-04, -9.10527452e-05,  2.45855439e-03]])

In [14]:
utils.block_report(rec, marker_data, marker_xyz)
utils.accuracy_report(
    rec, marker_xyz, gt, "after GCP bundle", camera_priors_used=USE_CAMERA_PRIORS
)


BLOCK QUALITY
  images            39 registered / 39 in block | 1 camera(s) SIMPLE_PINHOLE
  intrinsics        [5865.135 3008.    2008.   ]
  points3D          43,749      observations 221,660
  track length      mean 5.07   median 4
                    2: 1,114 (2.5%)   3-5: 30,330 (69.3%)   6-9: 8,254 (18.9%)   10+: 4,051 (9.3%)
  observations/img  min 3,414   mean 5,684   max 7,026

  tie-point reprojection error (px)
    mean 0.607   median 0.411   RMS 0.816   p90 1.456   p99 2.261   max 3.98
    > 2 px: 1,109 points (2.5%)

  redundancy        443,320 measurements - 131,484 unknowns  ->  r = 311,836

  marker observations (156 in 39 images)
    reprojection  mean 0.07 px   median 0.06   max 0.19
    worst:  gcp.018 0.18 px,  gcp.007 0.17 px,  gcp.023 0.14 px,  gcp.008 0.14 px,  gcp.010 0.10 px

ACCURACY vs GROUND TRUTH - after GCP bundle
                          bias E   bias N   bias U    RMSE 3D
                            (cm)     (cm)     (cm)       (cm)
  camera centres n=3

## 4. Export

Binary + text model for COLMAP, plus a point cloud carrying track length and
reprojection error per point so the sparse cloud can be filtered in a viewer.

In [16]:
utils.export_reconstruction(rec, sfm_gcp_path)
utils.save_point_cloud_as_ply(rec, sfm_gcp_path / "sparse_gcp_o3d.ply")

  wrote bin + txt + ply to /home/francesco/dev/pycolmap_ba/data/ENRICH-Aerial/nadir-2/colmap/sparse_gcp
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


True